# The `mgnipy.MGnipy()` client

Here we provide additional information about the `mgnipy.MGnipy()` client

```{margin}
After clicking the "Activate Notebook" button you can run the cells in this browser. Alternatively, you can also click on the 🚀 to launch in colab or binder. 
```
<button title="Make live" style="display:inline-flex;align-items:center;gap:0.4rem;padding:0.5rem 1rem;border:0;border-radius:20px;background:linear-gradient(135deg,#0f766e,#14b8a6);color:white;cursor:pointer;font-size:1rem;" class="thebe-button" onclick="initThebeSBT()">Activate Notebook</button>

---

In [1]:
# uncomment below if colab
#!pip install mgnipy

## Why start with the `mgnipy.MGnipy` client?

- **Unified configuration:** Central `MGnipyConfig` for base URL, credentials, token handling, and cache settings — one place to change behavior

- **Client as a context manager**: Helps make sure that any connections are closed via `with` block also has helpers for checking status `.status` and to `.close()`/`.aclose()` 

- **Tidier cache invalidation:** All the cache files across all resource endpoints (e.g., `MG.studies`, `MG.analysis`, `MG.biome`) go to a consistent place. The `MG.clear_subcaches()` can then clear all the mgnipy cache files for all different requests made.



## Quick to start

You can create a single `MGnipy()` instance and then access resource proxies from it. Those resource proxies aka resource endpoint-specific `MGnifier()`s would then share the same configuration of `MGnipy()`

In [2]:
from mgnipy import MGnipy

# Create a default client (will pick up .env if present)
MG = MGnipy(cache_dir="temp_example")

# details
print(MG)

# check if there is an active session (shouldnt be one)
MG.status

MGnipy(config=api_version=<SupportedApiVersions.V2: 'v2'> base_url=HttpUrl('https://www.ebi.ac.uk/') cache_dir=PosixPath('temp_example'))
Client or AuthenticatedClient type: Client
Owned by this instance: True
HTTP client open: False
Async client open: False



We can then easily point to the different MGnify API endpoints

In [3]:
# for example we can access the samples MGnify resource
samples = MG.samples

# some info
print(samples)

# more info
samples.describe_endpoint()

MGnifier instance for resource: samples
I.e., mgnipy.V2.proxies.samples.Samples
----------------------------------------
Base URL: https://www.ebi.ac.uk/
Parameters: {}
Example request URL: https://www.ebi.ac.uk/metagenomics/api/v2/samples?page=1
Endpoint module: mgnipy.emgapi_v2_client.api.samples.list_mgnify_samples
Is list endpoint (returns paginated results): True
Cache directory: temp_example/3fddd8853bdd0204eeaeda6c5b9b42b48c8a25ca4f034132d94eb1f93e01ac48

List all samples analysed by MGnify

MGnify samples inherit directly from samples (or BioSamples) in ENA.

Supported parameters:
- biome_lineage: None | str | Unset The lineage to match, including all descendant biomes
- search: None | str | Unset Search within sample titles and accessions
- order: ListMgnifySamplesOrderType0 | None | Unset
- page: int | Unset Default: 1.
- page_size: int | None | Unset


## Client as Context Manager

> Context managers allow you to allocate and release resources precisely when you want to. The most widely used example of context managers is the with statement. ...
[Read more here](https://book.pythontips.com/en/latest/context_managers.html)

MGnipy will take care of closing the clients if you use `with` blocks -- alternatively you can `.close()` manually 

For example:

In [4]:
# small query to get 3 per page 
modified_search = samples.filter(page_size=3)

with MG: 
    # within this client context, get 3 pages of samples resource
    modified_search.bulk_fetch(limit=2)

modified_search.metadata.to_pandas(expand_nested_dicts=True)

Retrieving samples pages:   0%|          | 2/143435 [00:01<28:07:45,  1.42it/s]


,accession,ena_accessions,sample_title,updated_at,biome__biome_name,biome__lineage
0,SAMEA2812786,"[SAMEA2812786, ERS565182]",C7~5,2026-04-30T12:08:10.969000+00:00,NaN,NaN
1,SAMEA113540503,"[ERS15536924, SAMEA113540503]",Study_5277_DNA,2026-05-01T07:31:37.685000+00:00,Fecal,root:Host-associated:Human:Digestive system:La...
2,SAMEA113539431,"[SAMEA113539431, ERS15535852]",Study_1322_RNA,2026-05-01T07:43:44.718000+00:00,Fecal,root:Host-associated:Human:Digestive system:La...
3,SAMN35300045,"[SAMN35300045, SRS17790516]",None,2026-05-05T14:11:42.920000+00:00,Fecal,root:Host-associated:Human:Digestive system:La...
4,SAMN35300038,"[SRS17790504, SAMN35300038]",None,2026-05-05T14:11:43.706000+00:00,Fecal,root:Host-associated:Human:Digestive system:La...
5,SAMN35300035,"[SAMN35300035, SRS17790505]",None,2026-05-05T14:11:44.229000+00:00,Fecal,root:Host-associated:Human:Digestive system:La...


we can check the status of the client to be sure

In [5]:
MG.status

# also can manuallly close 
# MG.close()

Client or AuthenticatedClient type: Client
Owned by this instance: True
HTTP client open: False
Async client open: False



## API helpers

We can also learn more about the MGnify API using the `mgnipy.MGnipy` client.

- `MG.list_resources()` returns the available endpoint names. 

- `MG.describe_resource()` to read parameter docs extracted from the OpenAPI spec.


In [6]:
# List known resources (strings like 'samples', 'studies', 'analyses')
print(MG.list_resources())

# Describe a resource
MG.describe_resources(MG.list_resources()[0])

['analyses', 'analysis', 'assemblies', 'assembly', 'genomes', 'genome', 'publications', 'publication', 'samples', 'sample', 'studies', 'study', 'runs', 'run', 'biomes', 'biome', 'miscellaneous', 'catalogues', 'catalogue', 'private_studies']
List all analyses (MGYAs) available from MGnify

Each analysis is the result of a Pipeline execution on a reads dataset (either a raw read-run, or an
assembly).

Supported parameters:
- page: int | Unset Default: 1.
- page_size: int | None | Unset


## Quick cleanup 

The `MG.clear_subcaches()` will clear all the mgnipy cache files, no matter if they were from `MG.studies` vs. `MG.analysis` vs. `MG.biome` etc, in the universal `MG.cache_dir`.


In [7]:
MG.clear_subcaches()